# Example 5.4.2 / 1: Growth Across Periods

Companion notebook for **5.4.2 Recursive Definitions** in the AMPL lab Documentation, section "Growth Across Periods". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Growth Across Periods

The example starts with demand in the first period and grows it by a fixed factor. In Python,
`to_dict()` retrieves the computed series.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PERIODS ordered;
param initial_demand >= 0;
param growth_factor >= 0;

param demand {t in PERIODS} >= 0 =
    if t = first(PERIODS) then initial_demand
    else demand[prev(t)] * growth_factor;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PERIODS"] = ["Q1", "Q2", "Q3", "Q4"]
ampl.param["initial_demand"] = 100
ampl.param["growth_factor"] = 1.1

In [5]:
# run.py
for period, value in ampl.param["demand"].to_dict().items():
    print(f"{period}: {value:.1f}")

Q1: 100.0
Q2: 110.0
Q3: 121.0
Q4: 133.1


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PERIODS := Q1 Q2 Q3 Q4;
param initial_demand := 100;
param growth_factor := 1.1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display demand;

demand [*] :=
Q1  100
Q2  110
Q3  121
Q4  133.1
;

